해당 코드는 아래와 같은 모델을 적용하여 진행해볼 것입니다.

허깅페이스 모델 :<br>
jhgan/ko-sroberta-multitask <br>
을 기반으로 Chroma DB의 Embedding Model 로 적용해서 전처리된 csv 파일로 확인

In [1]:
# 필요한 라이브러리 설치
!pip install -q sentence_transformers chromadb tiktoken transformers langchain pypdf langchain-community torch accelerate bitsandbytes

In [2]:
# 필요한 라이브러리 임포트
import os
import tiktoken
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

In [3]:
# 토큰화 함수 정의
tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

In [4]:
# PDF 로드
loader = PyPDFLoader("/content/drive/MyDrive/mulcam_bigdata/3조 : 최종 프로젝트/김성원/Data/헬스케어데이터/응급처치.pdf")
pages = loader.load_and_split()

# 텍스트 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = text_splitter.split_documents(pages)

# T4 기준 8초

In [5]:
# 임베딩 모델 설정
embed_model_name = "jhgan/ko-sroberta-multitask"
embed_model_kwargs = {'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
embed_encode_kwargs = {'normalize_embeddings': True}
hf_embeddings = HuggingFaceEmbeddings(
    model_name=embed_model_name,
    model_kwargs=embed_model_kwargs,
    encode_kwargs=embed_encode_kwargs
)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10

In [6]:
# Chroma DB 생성
db = Chroma.from_documents(docs, hf_embeddings)

# T4 기준 23초

In [7]:
# # 질문-답변 파이프라인 설정 (HuggingFace 모델 사용)
# qa_model_name = "beomi/KoAlpaca-Polyglot-5.8B"
# qa_pipeline = pipeline("question-answering", model=qa_model_name)

# def get_answer(query, context):
#     result = qa_pipeline(question=query, context=context)
#     return result['answer']

# # 세션이 종료됨

In [10]:
# # LLAMA 3 모델 설정
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name,
#                                              device_map="auto",
#                                              torch_dtype=torch.float16,
#                                              load_in_8bit=True)

In [14]:
from langchain.llms import HuggingFacePipeline

# 대체 모델 설정 (예: OPT 모델)
model_name = "facebook/opt-1.3b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map="auto",
                                             torch_dtype=torch.float16,
                                             load_in_8bit=True)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [15]:
# 파이프라인 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=2048,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15
)

In [16]:
# LangChain용 LLM 객체 생성
llm = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


In [17]:
# RetrievalQA 체인 설정
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(),
    return_source_documents=True
)

In [18]:
# LangChain용 LLM 객체 생성
llm = HuggingFacePipeline(pipeline=pipe)

In [19]:
# 질문-답변 실행
query = "응급처치 시 지켜야하는 원칙은?"
try:
    result = qa_chain.invoke({"query": query})
    print("질문:", query)
    print("답변:", result["result"])
    print("참고 문서:", result["source_documents"][0].page_content)
except Exception as e:
    print(f"오류 발생: {e}")

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


질문: 응급처치 시 지켜야하는 원칙은?
답변: Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

502생애주기별 안전교육 길잡이 지도서
또한 잘못된 처치나 이송은 환자의 상태를 악화시킬 수도 있어 기본적인 지식이 필요하나, 맥박과 
호흡이 없어 심폐소생술이 필요한 환자는 비록 약간의 잘못이 있더라도 인명 소생술을 시행하는 
것이 아무것도 안 하는 것보다 환자를 살릴 가능성이 높아진다.
(5) 응급처치 시 지켜야 할 원칙
응급처치 시행 시 준수해야 할 원칙이 있는데 이는 다음과 같다.
①  응급처치를 하는 사람 자신부터 안전을 확보해야 한다. 구조자가 위험한 상태에서 환자에게 
달려드는 것은 양쪽 다에게 해로운 일이다.
②  언제나 신속, 침착, 질서 있게 대처해야 한다.③  여러 환자가 있는 경우 긴급한 환자부터 처치해야 한다. 이 때 누가 긴급한 환자인지 구별하는 
중증도 분류가 필요할 수 있다.
④  이송이 필요한 상태라면 지체 없이 119에 신고하여 도움을 받아야 한다.
⑤  음식물을 줄 때는 신중을 기한다. 특히 무의식 환자에게 음식물 제공은 기도를 막아 숨을

505www.mpss.go.kr
제 2 장  I 생애주기별 안전교육정리하기 1
응급처치의 개념과 원리
① 동영상 자료 사연(응급처치 내용) 후 응급처치의 개념 및 원리를 정리해 보자.
2. 인명소생술
지도상의 유의점
사람은 누구나 한 번 죽게 마련이다. 그러나 갑작스러운 응급 상황을 당했을 때 올바른 조치를 
취한다면 그 중 일부는 죽음을 맞이하지 않을 수도 있다. 그 올바른 조치 중 가장 기본적인 방법이 
인명소생술이고 일반 시민, 학생으로서 알아야 하는 필수적인 기술이다. 이 장에서는 사람의 삶에서 
가장 기본적인 생명 유지에 대한 기술을 익히게 된다.